A poverty line that means something: 2,900 kcal per adult equivalent
====================================================================

**Author:** Ethan Ligon



## What this is



Session 2's poverty line was the lower quartile of food purchases, chosen
so that the unweighted headcount would come out at 0.25 and the code could
be checked against it.  That is a test, not a line.  A line means
something when it is the cost of a bundle that does something, and the
convention since Ravallion (1994) is that the bundle delivers a stated
number of calories, priced the way poor households actually buy them.

This notebook builds that line for GLSS7: the food bundle of households in
the second and third deciles, priced at national median unit values, scaled
to 2,900 kcal per adult equivalent per day.  Then it recomputes
$P_0, P_1, P_2$ and finds that the interesting number is not the
headcount but the scale factor, which tells you how much of what people
eat the diary never saw.

-   **Prerequisites:** `lsms_library` on the release kernel, GhanaLSS microdata.
    Self-contained; it doesn't assume `session2.ipynb` has run.  The
    `equivalence_scales` notebook explains the adult-equivalent scale used
    here; it's restated below in four lines.



## Setup



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ghana = ll.Country('GhanaLSS')
wave = '2016-17'

food_total = ghana.food_expenditures().groupby(['t', 'i']).sum().squeeze()
sample = ghana.sample()

C = food_total.xs(wave, level='t')                 # cedi per household, purchases
s = sample.xs(wave, level='t').reindex(C.index)
w, region = s.weight, s.strata

# Adult equivalents: under-14s count CHILD, everyone else counts one.
hc = ghana.household_characteristics().xs(wave, level='t').droplevel('v')
cells    = [c for c in hc.columns if c[:2] in ('F ', 'M ')]
children = [c for c in cells if c.split()[1] in ('00-03', '04-08', '09-13')]
CHILD = 0.5
A = (hc[[c for c in cells if c not in children]].sum(axis=1)
     + CHILD * hc[children].sum(axis=1)).reindex(C.index)
c = C / A                                          # cedi per adult equivalent

def fgt(c, w, z, alpha=0):
    """Weighted FGT_alpha.  Sums over the poor only: see session 2 for why."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w)
    c, w = c[ok], w[ok]
    poor = c < z
    return np.sum(w[poor] * ((z - c[poor]) / z) ** alpha) / np.sum(w)

z0 = c.quantile(0.25)                              # session 2's placeholder, per AE
print(f"placeholder line z0 = {z0:.2f} per adult equivalent")
for a in (0, 1, 2):
    print(f"  P_{a} = {fgt(c, w, z0, a):.4f}")

## 1.  The reference group



Ravallion's argument for a reference group is that the bundle should be
one poor households actually choose, so that the line prices their
preferences rather than a nutritionist's.  Deciles 2 and 3 of consumption
per adult equivalent, unweighted, keep the very poorest out (their bundles
are the most likely to be mis-recorded) and stay well below the median.



In [1]:
decile = pd.qcut(c, 10, labels=False) + 1
ref = c.index[decile.isin([2, 3])]

print(f"{len(ref)} reference households; "
      f"purchases per AE {c[ref].mean():.1f}, "
      f"mean adult equivalents {A[ref].mean():.2f}")

**Exercise 1.1.** The deciles are unweighted, so the reference group is a
quarter of the *sample*, not of the population.  Redo them with the survey
weights.  How different is the group, and does it matter for the line?



## 1.  The bundle, at national prices



Everything the reference households acquired, purchased or grown, by item
and unit.  Priced at the national median unit value from purchases, and
where nobody buys an item in that unit, at the median price households put
on their own production.  Only items with an energy content enter: the
`food_quantities` table converts to kilograms where the library has a
conversion, and `nutrition()` is built from those kilograms, so an item
that never converts (cooked rice bought by value, say) carries cedi but no
kilocalories and would put cost in the numerator with nothing in the
denominator.



In [1]:
acq = ghana.food_acquired().xs(wave, level='t')
pu = acq.xs('purchased', level='s')

p0 = ((pu.Expenditure / pu.Quantity).replace([np.inf, -np.inf], np.nan).dropna()
        .groupby(['j', 'u']).median())
pp = acq.xs('produced', level='s').Price.groupby(['j', 'u']).median()
price = p0.combine_first(pp)                       # national median, per item-unit

fq = ghana.food_quantities().xs(wave, level='t')
kcal_items = fq.xs('kg', level='u').index.get_level_values('j').unique()

q = acq.Quantity.groupby(['i', 'j', 'u']).sum()    # both sources, both windows
q = q[q.index.get_level_values('i').isin(ref) & q.index.get_level_values('j').isin(kcal_items)]
cost = (q * price.reindex(q.index.droplevel('i')).values).groupby('i').sum()

E_ref = pu.Expenditure[pu.index.get_level_values('i').isin(ref)].groupby('j').sum()
print(f"{len(kcal_items)} of {acq.index.get_level_values('j').nunique()} items convert to kg; "
      f"they carry {100 * E_ref[E_ref.index.isin(kcal_items)].sum() / E_ref.sum():.1f}% "
      f"of the reference group's purchase expenditure")
print(f"bundle cost at national prices, per reference household: {cost.mean():.1f}")

171 of the 179 items convert, and they carry 92% of the reference group's
purchase expenditure.  The bundle at national prices costs 472 cedi per
reference household over the diary window.

**Exercise 2.1.** The cost above is the group's bundle valued at national
prices, and the group's actual expenditure is its bundle at the prices it
paid.  Compare the two.  If the poor pay less than the median for the same
item-unit, which way is the line biased?



## 1.  Calories, and the scale factor



`nutrition()` returns kilocalories per household over the diary window.
It counts own production: households with no purchases at all still carry
energy.  The window is six visits at five-day intervals, so `DAYS` is 30;
it's a parameter because the `recall_and_diaries` notebook shows the first
window is longer than that.



In [1]:
nut = ghana.nutrition().xs(wave, level='t').droplevel('v')
E = nut.Energy.reindex(ref)

DAYS, KCAL = 30, 2900
kcal_per_ae_day = E.sum() / A[ref].sum() / DAYS
cost_per_kcal = cost.sum() / E.sum()
z = cost_per_kcal * KCAL * DAYS                    # cedi per AE per DAYS

print(f"reference group: {kcal_per_ae_day:,.0f} kcal per adult equivalent per day")
print(f"cost per 1,000 kcal at national prices: {1000 * cost_per_kcal:.2f} cedi")
print(f"scale factor to {KCAL} kcal: {KCAL / kcal_per_ae_day:.2f}")
print(f"line z = {z:.2f} per adult equivalent per {DAYS} days (placeholder was {z0:.2f})")

The recorded bundle delivers 1,147 kcal per adult equivalent per day.
Scaling it to 2,900 multiplies its cost by 2.53, and the line comes out at
257 cedi per adult equivalent per 30 days against a placeholder of 55.
For a check outside the survey, the GLSS7 poverty report's extreme (food)
line is GHS 982.94 per adult equivalent per year in January 2017 prices,
about 81 per 30 days.  Ours is three times that, from the same survey.

Read the scale factor before the line.  It says that the diary, converted
to kilograms and summed, accounts for about two fifths of the calories a
household needs.  Some of that is real (food eaten away from home, gifts),
and some of it is bookkeeping: an item counts as convertible above if
*any* of its units converts, so a heap of tomatoes enters the cost and not
the calories.  A line scaled by two and a half is a line built on the part
of the bundle you can see, applied to all of it, and the factor of three
against the official line is mostly that.

**Exercise 3.1.** Which items drive the shortfall?  `nutrition()` is per
household, so you can't decompose it by item from here.  Take the reference
group's purchased kilograms from `food_quantities()` instead, and rank
items by kilograms per adult equivalent per day.  Is anything obviously
missing that a Ghanaian household eats every day?



## 1.  P\_0, P\_1, P\_2



Two versions of consumption.  Purchases per adult equivalent is what
session 2 used.  The line values a bundle that includes own production, so
the consistent aggregate includes it too, valued as in the `food_sources`
notebook.



In [1]:
value = acq.Expenditure.where(acq.Expenditure.notna(), acq.Quantity * acq.Price)
C_all = value.groupby('i').sum().reindex(C.index)
c_all = C_all / A

rows = {}
for name, cc in (('purchases per AE', c), ('purchases + own production per AE', c_all)):
    rows[name] = {f'P_{a}': fgt(cc, w, z, a) for a in (0, 1, 2)}
    rows[name]['P_0 at placeholder'] = fgt(cc, w, z0, 0)
pd.DataFrame(rows).T.round(4)

At the 2,900-kcal line the headcount on purchases is 0.785 and on
purchases plus own production 0.724; the placeholder gave 0.137 and 0.053.
On the fuller aggregate the gap is 0.333 and the squared gap 0.190.  These
are not poverty rates you'd publish.  They
are what you get when a line built from a partial bundle is scaled to a
full requirement, and the honest report is that the survey's food diary
supports a relative line and not this one.

**Exercise 4.1.** Suppose instead that the diary is right and the households
in deciles 2 and 3 really do get 1,147 kcal per adult equivalent.  What
would you conclude about them, and about the 2,900 target?  Look up where
2,900 comes from before answering.



## 1.  How much does the reference group matter?



The line is national prices times a bundle, and the bundle is the only
thing the reference group affects.  The textbook expectation is that
richer households buy dearer calories, so that a higher reference group
means a higher line for the same 2,900.



In [1]:
q_all = acq.Quantity.groupby(['i', 'j', 'u']).sum()
q_all = q_all[q_all.index.get_level_values('j').isin(kcal_items)]
cost_all = (q_all * price.reindex(q_all.index.droplevel('i')).values).groupby('i').sum()

per_decile = []
for d in range(1, 11):
    hh = c.index[decile == d]
    per_decile.append(1000 * cost_all.reindex(hh).sum() / nut.Energy.reindex(hh).sum())
per_decile = pd.Series(per_decile, index=range(1, 11), name='cedi per 1,000 kcal')
print(per_decile.round(2).to_string())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(per_decile.index, per_decile.values, marker='o', color='#1f4e79')
ax.axvspan(1.5, 3.5, color='#d95f0e', alpha=0.15, lw=0)
ax.annotate('reference group', (2.5, per_decile.max()), ha='center', va='top', color='#8c2d04')
ax.set_xlabel('decile of purchases per adult equivalent')
ax.set_ylabel('cedi per 1,000 kcal, national prices')
ax.set_title('Cost per calorie by decile')
ax.set_xticks(range(1, 11))
ax.spines[['top', 'right']].set_visible(False)
plt.show()

It barely does.  Cost per 1,000 kcal is 2.67 in the bottom decile and
3.01 in the top, and it is not monotone in between: deciles 2 to 4 pay
about 2.95, deciles 7 and 8 about 2.5.  The reference group's calories
are as dear as anyone's.  Moving it to deciles 7–8 would *lower* the line
by about 15%, and nothing else would change, since prices are national by
construction.  So here the choice of reference group is worth a sixth of
the line and the scale factor is worth a factor of 2.5; a calorie-based
line is only as objective as the calorie count it rests on.

**Exercise 5.1.** Compute the line and $P_0$ for reference groups 1–2,
2–3, 3–4 and 4–5.  Plot $P_0$ against the line.  Ravallion's
recommendation was to iterate until the reference group's mean consumption
equals the line; does that fixed point exist here, and where?



## Exercises



1.  The line was built on the full acquisition table and applied, in one
    row of section 4, to purchases alone.  Build a third aggregate: purchases
    plus own production *restricted to `kcal_items`*, so that consumption
    and line cover the same goods.  Report $P_0$ and say whether it's
    closer to the purchases row or the full row, and why.
2.  The 2,900 is per adult equivalent under a scale where a child is half
    an adult.  The GLSS reports use a calorie-requirement scale by age and
    sex.  Build one from any published requirement table, redo sections 3
    and 4, and report how far the headcount moves.  Then say which of the
    two scales is the right one to use with a calorie-based line, and why
    the question has a definite answer here when it didn't in the
    `equivalence_scales` notebook.
3.  Repeat the whole construction for GLSS6 (`2012-13`).  Nominal prices
    differ, so the lines won't be comparable in cedi; the scale factors in
    section 3 will be.  Did the diary see more or less of the bundle in
    2012-13, and what changed in the instrument between the rounds?

